# 03 — Model Comparison

Radar plots and bar charts comparing 6 local (Ollama) models vs GPT-4o and Claude 3.5 Sonnet.

**Sections:**
1. Overview bar chart — functional accuracy and security pass rate
2. Radar / spider chart — multi-dimensional capability profile
3. Model × condition matrix (functional accuracy)
4. Parameter efficiency: accuracy per billion parameters
5. Cost-performance frontier (API vs local)
6. Paired statistical tests — DeepSeek vs GPT-4o

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

_results_path = Path('../results/experiment_results.json')
DATA = json.loads(_results_path.read_text()) if _results_path.exists() else {}
print('Results loaded.' if DATA else 'Using paper reference values.')

# All 8 models with paper reference values (sc+rag+3r condition)
MODELS = pd.DataFrame([
    {'name': 'DeepSeek Coder v2', 'short': 'DeepSeek',  'params_b': 16,  'type': 'local',
     'func': 69.4, 'sec': 59.3, 'bp': 61.2, 'rounds': 2.3, 'time_s': 142},
    {'name': 'Llama 3.1',         'short': 'Llama3.1',  'params_b': 70,  'type': 'local',
     'func': 72.3, 'sec': 62.1, 'bp': 64.8, 'rounds': 2.1, 'time_s': 381},
    {'name': 'Qwen 2.5 Coder',    'short': 'Qwen2.5',   'params_b': 32,  'type': 'local',
     'func': 74.1, 'sec': 63.8, 'bp': 66.3, 'rounds': 2.0, 'time_s': 198},
    {'name': 'CodeLlama',         'short': 'CodeLlama', 'params_b': 13,  'type': 'local',
     'func': 57.8, 'sec': 51.2, 'bp': 52.9, 'rounds': 2.8, 'time_s': 98},
    {'name': 'Mistral',           'short': 'Mistral',   'params_b': 7,   'type': 'local',
     'func': 54.3, 'sec': 48.7, 'bp': 49.8, 'rounds': 2.9, 'time_s': 61},
    {'name': 'Phi-3',             'short': 'Phi-3',     'params_b': 3.8, 'type': 'local',
     'func': 49.1, 'sec': 43.6, 'bp': 44.2, 'rounds': 3.0, 'time_s': 38},
    {'name': 'GPT-4o',            'short': 'GPT-4o',    'params_b': None,'type': 'api',
     'func': 93.9, 'sec': 84.2, 'bp': 87.1, 'rounds': 1.4, 'time_s': 28},
    {'name': 'Claude 3.5 Sonnet', 'short': 'Claude3.5', 'params_b': None,'type': 'api',
     'func': 91.2, 'sec': 81.7, 'bp': 84.3, 'rounds': 1.6, 'time_s': 32},
])
print(MODELS[['short','type','func','sec','bp']].to_string(index=False))

## 1. Overview Bar Chart

In [ ]:
df = MODELS.sort_values('func', ascending=False)
x  = np.arange(len(df))
w  = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
colors_type = df['type'].map({'local': '#4472C4', 'api': '#ED7D31'})
b1 = ax.bar(x - w/2, df['func'], w, color=colors_type,       alpha=0.85, label='Functional accuracy')
b2 = ax.bar(x + w/2, df['sec'],  w, color=colors_type.values,alpha=0.55, label='Security pass rate')

ax.set_xticks(x)
ax.set_xticklabels(df['short'], fontsize=9, rotation=10)
ax.set_ylabel('Pass Rate (%, sc+rag+3r)')
ax.set_title('Model Comparison: Functional Accuracy & Security Pass Rate (sc+rag+3r)')
ax.set_ylim(35, 105)
ax.axhline(y=93.9, color='gray', linestyle=':', linewidth=1)
ax.text(len(df)-0.5, 94.5, 'GPT-4o ceiling', fontsize=8, color='gray')
ax.grid(axis='y', linestyle='--', alpha=0.4)

patches = [
    mpatches.Patch(color='#4472C4', label='Local (Ollama)'),
    mpatches.Patch(color='#ED7D31', label='API (commercial)'),
]
ax.legend(handles=patches + [
    mpatches.Patch(color='gray', alpha=0.6, label='Solid=Functional / Light=Security')
])
plt.tight_layout()
plt.show()

## 2. Radar / Spider Chart — Multi-Dimensional Capability Profile

In [ ]:
categories = ['Functional\nAccuracy', 'Security\nPass Rate', 'Best\nPractice',
               'SC\nEfficiency', 'Throughput\n(tasks/min)']
N = len(categories)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]

def radar_vals(row):
    sc_eff   = max(0, (4 - row['rounds']) / 3 * 100)   # 3r → 100%, 0r → 33%
    tput     = min(100, 60 / row['time_s'] * 100 * 3)   # normalized
    return [row['func'], row['sec'], row['bp'], sc_eff, tput]

highlight = ['DeepSeek', 'Qwen2.5', 'GPT-4o', 'Claude3.5']
colors_r  = ['#4472C4', '#70AD47', '#ED7D31', '#C00000']

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for model_short, color in zip(highlight, colors_r):
    row = MODELS[MODELS['short'] == model_short].iloc[0]
    vals = radar_vals(row)
    vals += vals[:1]
    ax.plot(angles, vals, color=color, linewidth=2, label=model_short)
    ax.fill(angles, vals, color=color, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 100)
ax.set_yticks([25, 50, 75, 100])
ax.set_yticklabels(['25', '50', '75', '100'], fontsize=7)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('Radar: Capability Profile (sc+rag+3r)', pad=20)
plt.tight_layout()
plt.show()

## 3. Model × Condition Matrix

In [ ]:
conditions = ['one-shot', 'one-shot+RAG', 'sc+3r', 'sc+rag+3r', 'sc+rag+5r']
models_m   = ['DeepSeek', 'CodeLlama', 'Mistral', 'Phi-3', 'Llama3.1', 'Qwen2.5', 'GPT-4o', 'Claude3.5']

# Functional accuracy matrix (rows=models, cols=conditions)
matrix = np.array([
    [52.1, 57.3, 63.8, 69.4, 71.2],  # DeepSeek
    [42.3, 46.8, 51.4, 57.8, 59.1],  # CodeLlama
    [38.9, 43.2, 48.1, 54.3, 55.7],  # Mistral
    [34.1, 38.4, 43.2, 49.1, 50.3],  # Phi-3
    [55.8, 60.7, 67.3, 72.3, 74.1],  # Llama3.1
    [57.4, 62.8, 69.1, 74.1, 75.8],  # Qwen2.5
    [81.3, 85.9, 91.2, 93.9, 94.3],  # GPT-4o
    [78.4, 83.1, 88.4, 91.2, 91.8],  # Claude3.5
])

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=30, vmax=100)
plt.colorbar(im, ax=ax, label='Functional Accuracy (%)')
ax.set_xticks(range(len(conditions)))
ax.set_xticklabels(conditions, rotation=15, ha='right', fontsize=9)
ax.set_yticks(range(len(models_m)))
ax.set_yticklabels(models_m)
for i in range(len(models_m)):
    for j in range(len(conditions)):
        ax.text(j, i, f'{matrix[i,j]:.0f}', ha='center', va='center', fontsize=8,
                color='black' if matrix[i,j] < 85 else 'white')
ax.set_title('Functional Accuracy (%) — Model × Condition Matrix')
plt.tight_layout()
plt.show()

## 4. Parameter Efficiency: Accuracy per Billion Parameters

In [ ]:
local = MODELS[MODELS['type'] == 'local'].copy()
local['eff_func'] = local['func'] / local['params_b']
local['eff_sec']  = local['sec']  / local['params_b']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, title in zip(axes,
                           ['eff_func', 'eff_sec'],
                           ['Functional Accuracy / B params', 'Security Pass Rate / B params']):
    bars = ax.bar(local['short'], local[col], color='#4472C4', alpha=0.85)
    for bar, val in zip(bars, local[col]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
                f'{val:.2f}', ha='center', fontsize=9)
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle('Parameter Efficiency (sc+rag+3r) — Local Models Only', fontsize=11)
plt.tight_layout()
plt.show()

best_func = local.loc[local['eff_func'].idxmax()]
best_sec  = local.loc[local['eff_sec'].idxmax()]
print(f'Most param-efficient (functional): {best_func["short"]} ({best_func["eff_func"]:.2f} pp/B)')
print(f'Most param-efficient (security):   {best_sec["short"]} ({best_sec["eff_sec"]:.2f} pp/B)')

## 5. Cost-Performance Frontier

In [ ]:
# Approximate cost per 300-task run (sc+rag+3r)
cost_data = pd.DataFrame([
    {'short': 'DeepSeek', 'func': 69.4, 'cost_usd': 0,   'type': 'local'},
    {'short': 'Llama3.1', 'func': 72.3, 'cost_usd': 0,   'type': 'local'},
    {'short': 'Qwen2.5',  'func': 74.1, 'cost_usd': 0,   'type': 'local'},
    {'short': 'CodeLlama','func': 57.8, 'cost_usd': 0,   'type': 'local'},
    {'short': 'Mistral',  'func': 54.3, 'cost_usd': 0,   'type': 'local'},
    {'short': 'Phi-3',    'func': 49.1, 'cost_usd': 0,   'type': 'local'},
    {'short': 'GPT-4o',   'func': 93.9, 'cost_usd': 5.2, 'type': 'api'},
    {'short': 'Claude3.5','func': 91.2, 'cost_usd': 6.1, 'type': 'api'},
])

fig, ax = plt.subplots(figsize=(8, 5))
colors_map = {'local': '#4472C4', 'api': '#ED7D31'}
for _, row in cost_data.iterrows():
    x_jitter = row['cost_usd'] + np.random.uniform(-0.03, 0.03)
    ax.scatter(row['cost_usd'], row['func'], s=100,
               color=colors_map[row['type']], zorder=3)
    ax.annotate(row['short'], (row['cost_usd'], row['func']),
                textcoords='offset points', xytext=(7, 3), fontsize=9)

ax.set_xlabel('Cost per 300-task run (USD)')
ax.set_ylabel('Functional Accuracy (%)')
ax.set_title('Cost-Performance Frontier: Local (free) vs API ($5–6 per run)')
ax.set_xlim(-0.5, 7.5)
ax.set_ylim(40, 100)
ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.text(0.55, 42, 'API cost →', color='gray', fontsize=8)
patches = [mpatches.Patch(color='#4472C4', label='Local / free'),
           mpatches.Patch(color='#ED7D31', label='API (~$5–6/run)')]
ax.legend(handles=patches)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()
print('Local best (Qwen 2.5 Coder): 74.1%  — gap to GPT-4o: 19.8pp')
print('Open-source reproduction cost: $0  |  API reproduction cost: ~$5–6 per run')

## 6. Paired Statistical Tests: DeepSeek vs GPT-4o

In [ ]:
import math

def paired_t_test(a, b):
    """Two-tailed paired t-test. Returns (t_stat, p_approx, cohens_d, ci_half)."""
    diffs = [x - y for x, y in zip(a, b)]
    n     = len(diffs)
    mean_d = sum(diffs) / n
    var_d  = sum((d - mean_d) ** 2 for d in diffs) / (n - 1)
    se     = math.sqrt(var_d / n)
    t      = mean_d / se if se > 0 else float('inf')
    # Approximation for df=4: p < 0.05 if |t| > 2.776
    p_str  = 'p < 0.05' if abs(t) > 2.776 else 'p ≥ 0.05'
    d      = mean_d / math.sqrt(var_d) if var_d > 0 else 0
    ci_h   = 2.776 * se  # 95% CI half-width
    return t, p_str, d, ci_h

# 5 experimental runs (seeds 42-46) — functional accuracy
deepseek_runs = [68.7, 69.8, 70.1, 69.1, 69.3]
gpt4o_runs    = [93.1, 94.2, 94.6, 93.7, 94.1]

# Use actual data if available
main = DATA.get('main_results', {})
if 'sc_rag_3r_deepseek_runs' in main:
    deepseek_runs = main['sc_rag_3r_deepseek_runs']
if 'sc_rag_3r_gpt4o_runs' in main:
    gpt4o_runs = main['sc_rag_3r_gpt4o_runs']

t, p_str, d, ci_h = paired_t_test(gpt4o_runs, deepseek_runs)

print('Paired t-test: GPT-4o vs DeepSeek Coder v2 (functional accuracy, 5 seeds)')
print(f'  GPT-4o:   {sum(gpt4o_runs)/len(gpt4o_runs):.1f}% ± {ci_h:.1f}pp (95% CI)')
print(f'  DeepSeek: {sum(deepseek_runs)/len(deepseek_runs):.1f}% ± {ci_h:.1f}pp (95% CI)')
print(f'  t = {t:.3f},  {p_str},  Cohen\'s d = {d:.2f}')
print(f'  Mean difference: {sum(gpt4o_runs)/len(gpt4o_runs) - sum(deepseek_runs)/len(deepseek_runs):.1f}pp')

# Visualize
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([deepseek_runs, gpt4o_runs], labels=['DeepSeek\nCoder v2', 'GPT-4o'],
           patch_artist=True,
           boxprops=dict(facecolor='#4472C4', alpha=0.6),
           medianprops=dict(color='black', linewidth=2))
ax.set_ylabel('Functional Accuracy (%, 5 seeds)')
ax.set_title(f'DeepSeek vs GPT-4o  ({p_str}, d={d:.2f})')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()